In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

In [ ]:
#一个 inception 块，总共有四个路径，最后按通道进行合并
class Inception(nn.Module):
    def __init__(self, in_channel, c1, c2, c3, c4, **kwargs):
        super().__init__(Inception, **kwargs)
        #第一个路径：1 * 1 卷积层
        self.L1_1 = nn.Conv2d(in_channel, c1, kernel_size=1)
        #第二个路径：1 * 1 卷积层 -> 3 * 3 卷积层
        self.L2_1 = nn.Conv2d(in_channel, c2[0], kernel_size=1)
        self.L2_2 = nn.Conv2d(c2[0], c2[1], kernel_size=3, padding=1)
        #第三个路径：1 * 1 卷积层 -> 5 * 5 卷积层
        self.L3_1 = nn.Conv2d(in_channel, c3[0], kernel_size=1)
        self.L3_2 = nn.Conv2d(c3[0], c3[1], kernel_size=5, padding=2)
        #第四个路径：3 * 3 池化层 -> 1 * 1 卷积层
        self.L4_1 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
        self.L4_2 = nn.Conv2d(in_channel, c4, kernel_size=1)

    def forward(self, X):
        #分别获得不同路径的输出结果
        out1 = F.relu( self.L1_1(X) )
        out2 = F.relu( self.L2_2( F.relu( self.L2_1(X) ) ) )
        out3 = F.relu( self.L3_2( F.relu( self.L3_1(X) ) ) )
        out4 = F.relu( self.L4_2( F.relu( self.L4_1(X) ) ) )
        #将输出按通道合并（dim = 1）
        return torch.cat((out1, out2, out3, out4), dim=1)

**下面是 GoogLeNet 的模型结构**

In [ ]:
#第一模块：7 * 7卷积 + 3 * 3池化
#通道：1 -> 64
block1 = nn.Sequential(
    nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
)

In [ ]:
#第二模块：1 * 1卷积 + 3 * 3卷积 + 3 * 3池化
#通道：64 -> 64 -> 192
block2 = nn.Sequential(
    nn.Conv2d(64, 64, kernel_size=1),
    nn.ReLU(),
    nn.Conv2d(64, 192, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
)

In [ ]:
#第三模块：2 个串联的 Inception 块 + 3 * 3池化
#通道：192 -> 256 -> 480
block3 = nn.Sequential(
    Inception(192, 64, (96, 128), (16, 32), 32),
    Inception(256, 128, (128, 192), (32, 96), 64),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
)

In [ ]:
#第四模块：5 个串联的 Inception 块 + 3 * 3池化
#通道：480 -> 512 ... 512 -> 528 -> 832
block4 = nn.Sequential(
    Inception(480, 192, (96, 208), (16, 48), 64),
    Inception(512, 160, (112, 224), (24, 64), 64),
    Inception(512, 128, (128, 256), (24, 64), 64),
    Inception(512, 112, (144, 288), (32, 64), 64),
    Inception(528, 256, (160, 320), (32, 128), 128),
    nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
)

In [ ]:
#第五模块：2 个串联的 Inception 块 + 全局平均池化（1 * 1输出）+ 展平层 + 全连接线性层
#通道：832 -> 1024 -> 10 -> 输出类别 10 维向量
block5 = nn.Sequential(
    Inception(832, 256, (160, 320), (32, 128), 128),
    Inception(832, 384, (192, 384), (48, 128), 128),
    nn.AdaptiveAvgPool2d((1,1)),
    nn.Flatten(),
    nn.Linear(1024, 10)
) 

In [ ]:
net = nn.Sequential(
    block1,
    block2,
    block3,
    block4,
    block5
)

In [ ]:
lr, num_epochs, batch_size = 0.1, 10, 128
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size, resize=96)
d2l.train_ch6(net, train_iter, test_iter, num_epochs, lr, d2l.try_gpu())